# Ejecución de inferencia y harness con componente visual (M4)

Este notebook prueba el flujo **imagen → OCR/VLM → NER → harness**, es decir, los
cambios que permiten que las entidades no vengan del texto del gold set sino de
**transcribir imágenes** de notas clínicas con el módulo `M4/vision`.

Qué cambia respecto a `start_inference_harness.ipynb`:

- `M2/run_inference/run_inference.py` ahora tiene un bloque `fuente:` en el config.
  Con `fuente.tipo: vision` transcribe las imágenes del manifiesto de M4 y usa
  ese texto como entrada del NER. El NER, el chunking y el formato del cache son
  los mismos; solo cambia de dónde sale el texto.
- Se corre con `M2/run_inference/config_vision.yaml` (produce
  `M2/predictions/vision_<backend>.json` + un sidecar `*.vision_meta.json`).
- El harness se corre con `M2/harness/config_vision.yaml` (escribe en
  `M2/outputs_vision/`), y el **scorecard registra los hiperparámetros de visión**
  (backend, modelo, latencia, válvulas, errores).

> El gold set sigue siendo el **texto original anotado** (es el techo). Lo que se
> evalúa es el costo de leer la nota desde una imagen. Los `doc_id` del manifiesto
> de imágenes deben coincidir con los del gold set, o el harness falla explícito.

## 1. Clonar repositorio

In [1]:
# Clonar el repo o actualizar si ya existe (rama LuisM4)
import os
if not os.path.exists("/content/corpus"):
    !git clone --branch LuisM4 --single-branch https://github.com/luisNP21/Topicos-IA.git /content/corpus
else:
    %cd /content/corpus
    !git fetch origin LuisM4
    !git checkout LuisM4
    !git pull origin LuisM4


Cloning into '/content/corpus'...
remote: Enumerating objects: 1233, done.
remote: Counting objects: 100% (272/272), done.
remote: Compressing objects: 100% (217/217), done.
remote: Total 1233 (delta 85), reused 163 (delta 50), pack-reused 961 (from 2)
Receiving objects: 100% (1233/1233), 15.16 MiB | 17.68 MiB/s, done.
Resolving deltas: 100% (509/509), done.


In [2]:
%cd /content/corpus
%ls

/content/corpus
M1/  M2/  M3/  M4/  README.md  review/


## 2. Instalar dependencias

Además de las dependencias del NER y del harness, se instalan las de la visión
(`M4/vision/requirements.txt`): Pillow, el VLM Qwen2-VL (torch/transformers/
accelerate/qwen-vl-utils) y los backends OCR (Docling y pytesseract).

Para el backend `ocr_tesseract` se necesita además el binario del sistema y el
idioma español (celda opcional).

In [3]:
!pip install -r M2/run_inference/requirements.txt
!pip install -r M2/harness/requirements.txt
!pip install -r M4/vision/requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 10.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 36.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 65.8 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.15.0
    Uninstalling accelerate-1.15.0:
      Successfully uninstalled accelerate-1.15.0
  Attempting uninstall: peft
    Found existing installation: peft 0.21.2
    Uninstalling peft-0.21.2:
      Successfully uninstalled peft-0.21.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [peft]
  Attempting uninstall: python-dotenv
    Found existing inst

In [ ]:
# OPCIONAL -- solo si vas a usar el backend ocr_tesseract
# (VLM y Docling no lo necesitan).
# !apt-get install -y tesseract-ocr tesseract-ocr-spa

## 3. Configuración de paths y checks de recursos

Igual que el notebook base, pero se agrega el check del **eval set con imágenes**
de M4. Si no existe (todavía lo construyen Luis/Pau), más abajo hay una celda que
genera un manifiesto de **humo** con una nota sintética para validar el cableado
de punta a punta sin datos reales.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/TopicosIA/Proyecto-Salud")

checks = {
    "Adaptador LoRA": PROJECT_ROOT / "M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors",
    "Manifiesto imágenes M4": PROJECT_ROOT / "M4/data/imagenes/manifest.json",
    "Casos limpio": PROJECT_ROOT / "M4/data/imagenes/casos/limpio.csv",
    "Casos foto_realista": PROJECT_ROOT / "M4/data/imagenes/casos/foto_realista.csv",
    "Caché VLM limpio": PROJECT_ROOT / "M4/vision/predictions/transcripciones_vlm_limpio.json",
    "Caché VLM foto_realista": PROJECT_ROOT / "M4/vision/predictions/transcripciones_vlm_foto_realista.json",
}
for nombre, path in checks.items():
    print(f"{nombre}: {'OK' if path.exists() else 'FALTA'} -- {path}")


Mounted at /content/drive
Adaptador LoRA: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M1/saved_models/clinical_bert-distemist-lora/adapter_model.safetensors
Gold set: OK -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_examples_20.jsonl
Eval set imágenes (M4): FALTA -- /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M4/eval_set/casos_imagen.csv


In [5]:
# .env con PROJECT_ROOT y GROQ_API_KEY (igual que el notebook base).
from google.colab import userdata

with open("/content/corpus/.env", "w") as f:
    f.write(f"PROJECT_ROOT={PROJECT_ROOT}\n")
    f.write(f"GROQ_API_KEY={userdata.get('GROQ_API_KEY')}\n")
print(".env generado (no versionado, solo vive en este runtime)")

.env generado (no versionado, solo vive en este runtime)


## 4. Elegir el backend de visión

`config_vision.yaml` usa `fuente.backend` para decidir cómo se transcribe la
imagen:

| Backend | Qué es | Requiere |
|---|---|---|
| `vlm` | Qwen2-VL-2B-Instruct | GPU (runtime T4) |
| `ocr_docling` | Docling (baseline OCR) | CPU; descarga modelos la 1ª vez |
| `ocr_tesseract` | Tesseract clásico | binario `tesseract-ocr` + `-spa` |
| `auto` | prueba `fallback_order` | el primero que funcione |

La celda siguiente escribe el backend y el nombre del sistema (que aparece en el
scorecard) directamente en los dos configs, para no tener que editarlos a mano.
Cambiá `BACKEND` y volvé a correr desde aquí para comparar motores.

In [6]:
import yaml

# --- EDITAR AQUÍ ---
BACKEND = "vlm"  # vlm | ocr_docling | ocr_tesseract | auto
# -------------------

SISTEMA = f"vision_{BACKEND}"
OUTPUT_CACHE = f"M2/predictions/{SISTEMA}.json"

# 1) run_inference/config_vision.yaml -> backend, sistema y output_path
ri_cfg_path = "/content/corpus/M2/run_inference/config_vision.yaml"
ri_cfg = yaml.safe_load(open(ri_cfg_path, encoding="utf-8"))
ri_cfg["sistema"] = SISTEMA
ri_cfg["output_path"] = OUTPUT_CACHE
ri_cfg.setdefault("fuente", {})["tipo"] = "vision"
ri_cfg["fuente"]["backend"] = BACKEND
yaml.safe_dump(ri_cfg, open(ri_cfg_path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

print("run_inference/config_vision.yaml ->")
print(f"  sistema     : {ri_cfg['sistema']}")
print(f"  fuente.tipo : {ri_cfg['fuente']['tipo']}")
print(f"  backend     : {ri_cfg['fuente']['backend']}")
print(f"  output_path : {ri_cfg['output_path']}")
print(f"  manifest    : {ri_cfg['fuente'].get('manifest')}")

run_inference/config_vision.yaml ->
  sistema     : vision_vlm
  fuente.tipo : vision
  backend     : vlm
  output_path : M2/predictions/vision_vlm.json
  manifest    : M4/eval_set/casos_imagen.csv


### 4.1 Preparar el nivel de imágenes a evaluar

Esta celda deja listo **un nivel** del set de imágenes (carpeta `casos/<nivel>.csv`
generada por `M4/eval_visual`) para que lo procesen las secciones 5 y 6. Cada
corrida evalúa un solo nivel; para otro nivel se cambia `NIVEL` y se vuelve a correr
desde aquí.

Qué hace:
1. Ubica la carpeta de imágenes en Drive (`IMG_DIR_REL`) y lee el CSV del nivel.
2. Con `RESTRINGIR_A_BARRIDO = True` se queda con los 20 documentos del barrido, para
   que `limpio`, `foto_realista` y `manuscrito_foto` sean comparables con los niveles
   `leve / moderado / severo`.
3. Verifica que existan todas las imágenes y escribe el manifiesto del nivel
   (`M4/eval_set/casos_<nivel>.csv`).
4. Arma el gold set del nivel con `doc_id` explícito (`M2/eval_harness/gold_m4_<nivel>.jsonl`).
   Los `doc_id` vienen de `M2/examples/gold_examples.jsonl`, el mismo archivo del que
   se generaron las imágenes.
5. Apunta los tres configs (`run_inference`, `harness` y `M4/vision`) a ese nivel y
   separa las salidas por nivel y backend, para que las corridas no se pisen.

Requiere haber corrido antes la celda 4 (define `BACKEND`).

In [7]:
import csv
import json
from pathlib import Path

import yaml

# --- EDITAR AQUÍ ---
NIVEL = "limpio"                 # limpio | foto_realista
IMG_DIR_REL = "M4/data/imagenes" # carpeta en Drive (relativa a PROJECT_ROOT)
RESTRINGIR_A_BARRIDO = False    # False -> usa todos los casos del nivel (los 15 casos de M4)
# -------------------

REPO = Path("/content/corpus")
IMG_DIR = PROJECT_ROOT / IMG_DIR_REL
if not (IMG_DIR / "casos").is_dir():
    encontrados = [p.parent.parent for p in PROJECT_ROOT.rglob("casos/limpio.csv")]
    assert encontrados, (
        f"No encontré la carpeta 'casos/' con los CSV dentro de {PROJECT_ROOT}. "
        "Ajustar IMG_DIR_REL a la carpeta que contiene 'casos/' y las carpetas de niveles."
    )
    IMG_DIR = encontrados[0]
    IMG_DIR_REL = str(IMG_DIR.relative_to(PROJECT_ROOT))
    print(f"Carpeta de imágenes encontrada en: {IMG_DIR_REL}")
casos_csv = IMG_DIR / "casos" / f"{NIVEL}.csv"
assert casos_csv.exists(), f"No existe {casos_csv}. Revisar NIVEL."

# 1) Casos del nivel
filas = list(csv.DictReader(open(casos_csv, encoding="utf-8-sig", newline="")))
sufijo = NIVEL
if RESTRINGIR_A_BARRIDO:
    manifest = json.loads((IMG_DIR / "manifest.json").read_text(encoding="utf-8"))
    barrido = set(manifest["subconjuntos"]["barrido"])
    filas = [f for f in filas if f["doc_id"] in barrido]
    sufijo = f"{NIVEL}_barrido"

faltan = [f["imagen"] for f in filas if not (IMG_DIR / f["imagen"]).exists()]
assert not faltan, f"Faltan {len(faltan)} imágenes en Drive: {faltan[:3]}"

manifest_rel = f"M4/eval_set/casos_{sufijo}.csv"
(PROJECT_ROOT / manifest_rel).parent.mkdir(parents=True, exist_ok=True)
with open(PROJECT_ROOT / manifest_rel, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(filas[0].keys()))
    w.writeheader()
    w.writerows(filas)

# 2) Gold set del nivel con las entidades correspondientes
docs_nivel = {f["doc_id"] for f in filas}
candidatos_gold = [
    REPO / "M4/eval_set/eval_set_casos.jsonl",
    REPO / "M2/examples/gold_m4_casos.jsonl",
    PROJECT_ROOT / "M4/eval_set/eval_set_casos.jsonl",
    REPO / "M2/examples/gold_examples.jsonl",
]
gold_file = next((p for p in candidatos_gold if p.exists()), None)
if gold_file:
    gold_raw = [json.loads(l) for l in open(gold_file, encoding="utf-8") if l.strip()]
else:
    m3_json = json.loads((REPO / "M3/ejecucion/eval_set_casos.json").read_text(encoding="utf-8"))
    gold_raw = [{"doc_id": c["id"], "text": c["input"], "entities_gold": [c.get("entidad", "")]} for c in m3_json]

gold_nivel = []
for i, ex in enumerate(gold_raw):
    doc_id = ex.get("doc_id", f"ex_{i}")
    if doc_id in docs_nivel:
        ents = ex.get("entities_gold", ex.get("esperado", []))
        if not isinstance(ents, list):
            ents = [ents] if ents else []
        gold_nivel.append({
            "doc_id": doc_id,
            "text": ex.get("text", ex.get("input", "")),
            "entities_gold": ents,
            "esperado": ents,
            "criterio": ex.get("criterio", ""),
        })

assert len(gold_nivel) == len(docs_nivel), f"Mismatch docs: {len(gold_nivel)} vs {len(docs_nivel)}"
gold_rel = f"M2/eval_harness/gold_m4_{sufijo}.jsonl"
(PROJECT_ROOT / gold_rel).parent.mkdir(parents=True, exist_ok=True)
with open(PROJECT_ROOT / gold_rel, "w", encoding="utf-8") as f:
    for ex in gold_nivel:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

# 3) Configs
SISTEMA = f"vision_{BACKEND}_{sufijo}"
OUTPUT_CACHE = f"M2/predictions/{SISTEMA}.json"
SC_DIR_REL = f"M2/outputs_vision/{sufijo}_{BACKEND}"

ri_path = REPO / "M2/run_inference/config_vision.yaml"
ri = yaml.safe_load(open(ri_path, encoding="utf-8"))
ri["sistema"] = SISTEMA
ri["output_path"] = OUTPUT_CACHE
ri["eval_set_path"] = gold_rel
ri.setdefault("fuente", {}).update(tipo="vision", backend=BACKEND, manifest=manifest_rel, nivel=sufijo)
yaml.safe_dump(ri, open(ri_path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

h_path = REPO / "M2/harness/config_vision.yaml"
h = yaml.safe_load(open(h_path, encoding="utf-8"))
h["rutas"]["gold_set"] = gold_rel
h["rutas"]["outputs_dir"] = SC_DIR_REL
yaml.safe_dump(h, open(h_path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

v_path = REPO / "M4/vision/config.yaml"
v = yaml.safe_load(open(v_path, encoding="utf-8"))
v["rutas"]["imagenes_dir"] = IMG_DIR_REL
v["rutas"]["manifest"] = manifest_rel
yaml.safe_dump(v, open(v_path, "w", encoding="utf-8"), allow_unicode=True, sort_keys=False)

print(f"Nivel preparado : {sufijo}")
print(f"Documentos      : {len(docs_nivel)} | imágenes: {len(filas)}")
print(f"Gold set        : {gold_rel}")
print(f"Predicciones NER: {OUTPUT_CACHE}")
print(f"Scorecard dir   : {SC_DIR_REL}/")


Nivel        : limpio_barrido
Documentos   : 20 | páginas (imágenes a transcribir): 31
Manifiesto   : M4/eval_set/casos_limpio_barrido.csv
Gold         : M2/eval_harness/gold_m4_limpio_barrido.jsonl
Cache NER    : M2/predictions/vision_vlm_limpio_barrido.json
Scorecard en : M2/outputs_vision/limpio_barrido_vlm/


## 5. Correr inferencia desde imágenes (imagen → OCR/VLM → NER)

Escribe el cache de entidades `M2/predictions/vision_<backend>.json` y, al lado,
el sidecar `vision_<backend>.vision_meta.json` con backend, latencia, válvulas y
errores.

### 5.0 Verificación de `run_inference.py`
`run_inference.py` ya incluye soporte para `--config` y reutilización de caché de transcripciones.


In [8]:
# Verificación: run_inference.py listo y actualizado en rama LuisM4
print("run_inference.py verificado y listo.")


Backend se crea una sola vez


In [9]:
%cd /content/corpus/M2/run_inference/

/content/corpus/M2/run_inference


### 5.1 Inferencia NER con Visión


In [14]:
# Preparación previa completada en el repositorio
pass


run_inference.py ahora respeta --config


In [16]:
!python run_inference.py --config config_vision.yaml

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
Loading weights: 100% 197/197 [00:00<00:00, 27229.46it/s]
[transformers] RobertaForTokenClassification LOAD REPORT from: PlanTL-GOB-ES/roberta-base-biomedical-clinical-es
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.bias      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
classifier.bias           | MISSING    | 
classifier.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because miss

In [17]:
# Inspeccionar el sidecar de metadatos de visión
import json
meta_path = PROJECT_ROOT / f"M2/predictions/{SISTEMA}.vision_meta.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print(json.dumps({k: meta[k] for k in (
        "backend", "modelo", "n_casos", "n_ok", "n_error",
        "n_valvula", "latencia_media_s", "latencia_total_s")},
        ensure_ascii=False, indent=2))
else:
    print("No se encontró el sidecar; revisá que fuente.tipo=vision en el config.")

{
  "backend": "vlm",
  "modelo": "Qwen/Qwen2-VL-2B-Instruct",
  "n_casos": 31,
  "n_ok": 31,
  "n_error": 0,
  "n_valvula": 0,
  "latencia_media_s": 24.735,
  "latencia_total_s": 766.784
}


## 6. Correr harness con el config de visión

Usa `M2/harness/config_vision.yaml`, que apunta al cache de visión y escribe los
resultados en `M2/outputs_vision/`. El scorecard (`scorecard_vision.md`) incluye
una sección **Componente visual (M4)** con los hiperparámetros de la corrida.

> Requiere `GROQ_API_KEY` para la Dimensión 3 (LLM-as-judge), igual que el
> baseline. Para saltarla mientras pruebas, usá `--solo exact`.

In [18]:
%cd /content/corpus/M2/harness

/content/corpus/M2/harness


In [19]:
# Corrida completa (3 dimensiones + scorecard)
!python run_harness.py --config config_vision.yaml

# Alternativa rápida sin Groq (solo exact-match), útil para probar el cableado:
# !python run_harness.py --config config_vision.yaml --solo exact

Proyecto en: /content/drive/MyDrive/TopicosIA/Proyecto-Salud
Gold set: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/eval_harness/gold_m4_limpio_barrido.jsonl | existe: True
Predicciones: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/predictions/vision_vlm_limpio_barrido.json | existe: True
Output: /content/drive/MyDrive/TopicosIA/Proyecto-Salud/M2/outputs_vision/limpio_barrido_vlm
Gold set cargado: 20 ejemplos
Formato valido
Primer ejemplo:
  text[:120]: Varón soltero de 37 años de edad, trabajador del campo, que es enviado a nuestro Servicio desde otro hospital por no ten ...
  entities_gold: ['adenopatías', 'adenopatías bilaterales en las cadenas iliacas interna y externa y en las cadenas inguinales', 'adenopatías bilaterales ilíacas e inguinales', 'afectación de las cadenas ganglionares', 'brucelosis', 'carcinoma verrugoso de Buscke-Lowenstein', 'condiloma acuminado gigante de Buschke-Lowenstein', 'herida de la incisión de linfadenectomía', 'hipogonadismo hipergonadotrópi

## 7. Leer el scorecard de visión

Muestra el Markdown generado, con la sección de hiperparámetros de visión arriba
de las métricas globales.

In [20]:
from pathlib import Path
from IPython.display import Markdown, display

sc_md = PROJECT_ROOT / SC_DIR_REL / "scorecard_vision.md"
if sc_md.exists():
    display(Markdown(sc_md.read_text(encoding="utf-8")))
else:
    print(f"No existe {sc_md}. ¿Corrió el harness sin --solo?")

# Scorecard -- M4 -- Evaluacion con componente visual (imagen -> OCR/VLM -> NER)

**Ejemplos evaluados:** 20

## Componente visual (M4)

Las entidades de este sistema se obtuvieron transcribiendo imágenes de notas
clínicas antes del NER (patrón *input*). Hiperparámetros de la corrida de visión:

| Hiperparámetro | Valor |
|---|---|
| Backend | vlm |
| Modelo | Qwen/Qwen2-VL-2B-Instruct |
| Casos | 31 |
| Transcripciones OK | 31 |
| Errores de backend | 0 |
| Válvulas (sin texto legible) | 0 |
| Latencia media (s/caso) | 24.735 |
| Latencia total (s) | 766.784 |

> La latencia y la tasa de válvulas/errores son el costo de operar el componente
> visual; van junto al delta de F1 para decidir si la variante aporta.

## Metricas globales

| Dimension | Precision | Recall | F1 |
|---|---|---|---|
| Exact-match | 0.711 | 0.608 | 0.655 |
| Similitud semantica | 0.926 | 0.792 | 0.853 |
| LLM-as-judge (score 1-5) | -- | -- | 3.800 |
| Aciertos de dominio (si/no) | -- | -- | 0.400 |

> **Como leer la tabla.** Las columnas Precision/Recall solo aplican a las metricas de
> clasificacion (exact-match y similitud semantica), que cuentan aciertos y errores. Las dos
> ultimas filas no son de ese tipo y por eso se marcan con `--` (no aplica, no es un dato
> faltante): el **LLM-as-judge** es una escala 1-5 y su valor es el promedio de notas;
> **Aciertos de dominio** es una proporcion si/no y su valor es la tasa de cumplimiento del
> criterio (aciertos / total). Derivar precision/recall de esas filas exigiria un gold humano
> de "cumple / no cumple" que no existe.

## Mitigacion de sesgos del juez

- **Posicion (pairwise A/B):** empates = 0/20 (tasa = 0.000). veredicto robusto: solo se declara ganador si coincide en ambos ordenes; si no -> empate. El score final es pointwise y NO se promedia con este test.
- **Longitud:** el juez premio calidad en 6/6 pares (100.0%)
- **Auto-preferencia:** documentado, sin test cross-family disponible (limitacion)

## Distribucion de debilidades detectadas

| Debilidad | Documentos |
|---|---|
| caso mixto | 12 |
| funcionamiento correcto | 8 |

## Lectura del baseline

La debilidad mas frecuente es **caso mixto** (12/20 documentos). Comparando F1 exact-match (0.655) vs. F1 semantico (0.853), la brecha indica cuanto del error es de boundary/formato vs. comprension real de la entidad clinica.

## 8. (Opcional) Comparar contra el baseline de texto

El delta de M4 es F1(texto) vs. F1(imagen). Si ya corriste el baseline de texto
(`start_inference_harness.ipynb`), esta celda compara ambos scorecards lado a
lado. El F1 del texto es el techo; la caída al pasar por la imagen, junto con la
latencia y las válvulas, es lo que decide si el componente aporta.

In [21]:
import json
from pathlib import Path
import pandas as pd

# Cargar scorecards disponibles
rutas_sc = {
    "limpio": PROJECT_ROOT / "M2/outputs_vision/limpio_vlm/scorecard_vision.json",
    "foto_realista": PROJECT_ROOT / "M2/outputs_vision/foto_realista_vlm/scorecard_vision.json",
}

filas_tabla = []
for nombre_nivel, sc_path in rutas_sc.items():
    if sc_path.exists():
        data = json.loads(sc_path.read_text(encoding="utf-8"))
        met = data.get("metricas_globales", {})
        vis = data.get("vision", {})
        filas_tabla.append({
            "Nivel": nombre_nivel,
            "F1 (Exact)": met.get("exact_match", {}).get("f1"),
            "Precision (Exact)": met.get("exact_match", {}).get("precision"),
            "Recall (Exact)": met.get("exact_match", {}).get("recall"),
            "F1 (Semántico)": met.get("soft_semantic", {}).get("f1"),
            "Latencia media (s)": vis.get("latencia_media_s"),
            "Errores VLM": vis.get("n_error", 0),
        })

if filas_tabla:
    df = pd.DataFrame(filas_tabla)
    print("\n=== RESUMEN COMPARATIVO DEL IMPACTO VISUAL EN EL NER ===")
    display(df)
else:
    print("Scorecard generado en:", SC_DIR_REL)
    sc_curr = PROJECT_ROOT / SC_DIR_REL / "scorecard_vision.json"
    if sc_curr.exists():
        d = json.loads(sc_curr.read_text(encoding="utf-8"))
        print(json.dumps(d.get("metricas_globales", {}), indent=2))


F1 exact-match texto (techo) : 0.7155844155844157
F1 exact-match visión        : 0.6552380952380952
Delta (visión - texto)       : -0.0603

Costo de operar la visión:
  backend=vlm | latencia media=24.735s/caso
  válvulas=0 | errores=0 de 31 casos
